# 项目V1.0：数据清洗与特征工程

**日期：** 2026-08-17 周一

## 今日目标

- 加载之前生成的缺陷数据 `defect_data.csv`
- 检查数据质量（缺失值、数据类型、类别分布）
- 将文字特征（缺陷类型、复现概率、严重程度）转换为数字
- 完成特征矩阵 X 和标签 y 的准备
- 划分训练集和测试集

## 第一步：加载并检查数据

In [3]:
import pandas as pd
import numpy as np

# 加载数据
df = pd.read_csv("defect_data.csv")
print("===== 数据概况 =====")
print(f"数据形状：{df.shape}")
print(f"\n前5行：")
display(df.head())

# 查看每列的数据类型
print("\n===== 数据类型 =====")
print(df.dtypes)

# 检查缺失值
print("\n===== 缺失值统计 =====")
print(df.isnull().sum())

# 查看严重程度分布
print("\n===== 严重程度分布 =====")
print(df["严重程度"].value_counts())

===== 数据概况 =====
数据形状：(500, 6)

前5行：


,缺陷类型,影响模块数,复现概率,数据影响,业务中断,严重程度
0,功能,1,必现,0,0,一般
1,界面,1,偶发,1,1,严重
2,性能,1,必现,0,1,严重
3,性能,5,必现,0,0,严重
4,功能,4,偶发,0,0,严重



===== 数据类型 =====
缺陷类型       str
影响模块数    int64
复现概率       str
数据影响     int64
业务中断     int64
严重程度       str
dtype: object

===== 缺失值统计 =====
缺陷类型     0
影响模块数    0
复现概率     0
数据影响     0
业务中断     0
严重程度     0
dtype: int64

===== 严重程度分布 =====
严重程度
严重    208
一般    155
致命    118
轻微     19
Name: count, dtype: int64


## 第二步：特征工程——将文字转换为数字
机器学习模型只能处理数字，所以需要把文字特征转换掉。

In [5]:
# ===== 2.1 处理标签（严重程度）=====
# 使用 LabelEncoder 把文字标签转成数字
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["严重程度_编码"] = label_encoder.fit_transform(df["严重程度"])

print("===== 标签编码 =====")
print("编码对应关系:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"{cls} -> {i}")

# ===== 2.2 处理文字特征 =====
# 缺陷类型和复现概率是分类特征，可以用 OneHotEncoder 或 LabelEncoder
# 对于缺陷类型，三种类别之间没有顺序关系，建议用 OneHotEncoder
# 对于复现概率，必现/偶发也没有顺序，但只有两类，可以用 LabelEncoder

# 方法一：使用 pd.get_dummies（推荐，简单直观）
df_encoded = pd.get_dummies(
    df,
    columns=["缺陷类型", "复现概率"],
    prefix=["类型", "复现"],
    drop_first=True   # 避免多重共线性
)

print("\n===== OneHot编码后的特征 =====")
print(df_encoded.columns.tolist())
display(df_encoded.head())

===== 标签编码 =====
编码对应关系:
一般 -> 0
严重 -> 1
致命 -> 2
轻微 -> 3

===== OneHot编码后的特征 =====
['影响模块数', '数据影响', '业务中断', '严重程度', '严重程度_编码', '类型_性能', '类型_界面', '复现_必现']


,影响模块数,数据影响,业务中断,严重程度,严重程度_编码,类型_性能,类型_界面,复现_必现
0,1,0,0,一般,0,False,False,True
1,1,1,1,严重,1,False,True,False
2,1,0,1,严重,1,True,False,True
3,5,0,0,严重,1,True,False,True
4,4,0,0,严重,1,False,False,False


**注意：** `drop_first=True` 会丢弃每个分类特征的第一类，避免冗余。比如“缺陷类型”原本有三种（功能/性能/界面），OneHot编码后变成3列，drop_first后变成2列（类型_功能、类型_性能），剩下那一列的信息被隐含了。

## 第三步：准备特征矩阵 X 和标签 y

In [7]:
# 删除原始文字列和标签列，只保留数值特征
X = df_encoded.drop(["缺陷类型_功能", "类型_性能", "严重程度", "严重程度_编码"], axis=1, errors="ignore")
# 实际上更简单的做法：
X = df_encoded.drop(["严重程度", "严重程度_编码"], axis=1)
y = df_encoded["严重程度_编码"]

print("===== 特征矩阵 =====")
print(f"X 形状：{X.shape}")
print(f"特征列名：{X.columns.tolist()}")
display(X.head(10))

print("\n===== 标签 =====")
print(f"y 形状：{y.shape}")
print(f"y 分布：")
print(y.value_counts())

===== 特征矩阵 =====
X 形状：(500, 6)
特征列名：['影响模块数', '数据影响', '业务中断', '类型_性能', '类型_界面', '复现_必现']


,影响模块数,数据影响,业务中断,类型_性能,类型_界面,复现_必现
0,1,0,0,False,False,True
1,1,1,1,False,True,False
2,1,0,1,True,False,True
3,5,0,0,True,False,True
4,4,0,0,False,False,False
5,5,1,0,False,False,True
6,4,0,0,False,False,False
7,5,0,0,False,True,True
8,5,0,1,True,False,False
9,3,1,0,True,False,True



===== 标签 =====
y 形状：(500,)
y 分布：
严重程度_编码
1    208
0    155
2    118
3     19
Name: count, dtype: int64


**确认特征列包含：**

- 影响模块数（数值）
- 数据影响（0/1）
- 业务中断（0/1）
- 类型_功能（0/1）
- 类型_性能（0/1）
- 复现_必现（0/1）

## 第四步：划分训练集和测试集

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("===== 数据划分 =====")
print(f"训练集大小：{X_train.shape}")
print(f"测试集大小：{X_test.shape}")
print(f"\n训练集标签分布：")
print(y_train.value_counts(normalize=True).round(3))
print(f"\n测试集标签分布：")
print(y_test.value_counts(normalize=True).round(3))

===== 数据划分 =====
训练集大小：(350, 6)
测试集大小：(150, 6)

训练集标签分布：
严重程度_编码
1    0.417
0    0.309
2    0.237
3    0.037
Name: proportion, dtype: float64

测试集标签分布：
严重程度_编码
1    0.413
0    0.313
2    0.233
3    0.040
Name: proportion, dtype: float64


## 第五步：标准化
分类模型中的 SVM、KNN、逻辑回归需要标准化。树模型（随机森林、决策树）不需要。为了统一处理，可以创建两个版本，或者只对需要的模型标准化。

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 查看标准化后的数据
print("===== 标准化后的训练集 =====")
print(f"均值（应接近0）：{X_train_scaled.mean(axis=0).round(3)}")
print(f"标准差（应接近1）：{X_train_scaled.std(axis=0).round(3)}")

===== 标准化后的训练集 =====
均值（应接近0）：[ 0.  0.  0.  0. -0.  0.]
标准差（应接近1）：[1. 1. 1. 1. 1. 1.]


## 今日验收标准

- 1. 成功加载 `defect_data.csv`，确认无缺失值
- 2. 文字特征已转换为数字（OneHot编码）
- 3. 特征矩阵 X 和标签 y 形状正确
- 4. 训练集和测试集已划分，且标签分布一致
- 5. 标准化完成，均值和标准差符合预期

## 2026-08-17 周一（第37天）项目V1.0：数据清洗与特征工程

### 完成内容
- 加载 defect_data.csv（500行×6列）
- 检查数据质量：无缺失值，类型正确
- 标签编码：严重程度 → 0/1/2/3
- OneHot编码：缺陷类型、复现概率
- 准备特征矩阵 X（6个特征）和标签 y
- 划分训练集（70%）和测试集（30%），使用 stratify=y
- 标准化（用于SVM/KNN/逻辑回归）

### 特征列
| 特征 | 类型 | 说明 |
|------|------|------|
| 影响模块数 | 数值 | 1-5 |
| 数据影响 | 0/1 | 是否影响数据 |
| 业务中断 | 0/1 | 是否业务中断 |
| 类型_功能 | 0/1 | 是否功能缺陷 |
| 类型_性能 | 0/1 | 是否性能缺陷 |
| 复现_必现 | 0/1 | 是否必现 |

### 数据划分
- 训练集：350条
- 测试集：150条
- stratify=y 保证类别比例一致

### 遇到的问题
- 暂无